## Sentiment analysis using RNN

In [1]:
# imports
import os
import kagglehub
import pandas as pd
import numpy as np

import transformers
from transformers import AutoTokenizer
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader



/Users/vip/soc/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Downloading Data
path = kagglehub.dataset_download(
    "abhi8923shriv/sentiment-analysis-dataset"
)
print(path)

device = torch.device("cuda" if torch.cuda.is_available()  else "mps" if torch.backends.mps.is_available() else "cpu")
print(device)

/Users/vip/.cache/kagglehub/datasets/abhi8923shriv/sentiment-analysis-dataset/versions/9
mps


In [3]:
# Getting Data for Train/test

train_df = pd.read_csv(os.path.join(path, "train.csv"),encoding='latin-1')
train_df = train_df[["selected_text", "sentiment"]]
train_df = train_df.dropna()

label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

train_df_text = train_df['selected_text'].tolist()
train_df_labels = train_df['sentiment'].tolist()

train_data = train_df_text[:int(0.8*len(train_df_text))]
train_labels = [label_map[label] for label in train_df_labels[ : int(0.8*len(train_df_labels)) ]]

test_data = train_df_text[int(0.8*len(train_df_text)):]
test_labels = [label_map[label] for label in train_df_labels[int(0.8*len(train_df_labels)) : ]]

In [4]:
# Tokenizing Data

train_text = []
test_text = []
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

encoded_train = tokenizer(train_data, padding=True, truncation=True, return_tensors="np")
encoded_test = tokenizer(test_data, padding=True, truncation=True, return_tensors="np")

train_data = encoded_train['input_ids']
test_data = encoded_test['input_ids']

print(train_data)

[[  101  1045  1036 ...     0     0     0]
 [  101 17111  2080 ...     0     0     0]
 [  101 18917  2033 ...     0     0     0]
 ...
 [  101  1049  3374 ...     0     0     0]
 [  101  6569   102 ...     0     0     0]
 [  101  2821  3398 ...     0     0     0]]


In [5]:
# Dataset class for Dataloader
class dataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return self.encodings[index], self.labels[index]

In [6]:
train_data = torch.tensor(train_data, dtype=torch.long, device=device)
train_labels = torch.tensor(train_labels, dtype=torch.long, device=device)

train_dataset = DataLoader(dataset(train_data, train_labels), batch_size=64, shuffle=True)
test_dataset = DataLoader(dataset(test_data, test_labels), batch_size=64, shuffle=False)

class RNN(nn.Module):
    def __init__(self, vocab_size, input_size, hidden_layer_size, output_size, pad_idx):
        super(RNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, input_size, padding_idx=pad_idx)
        self.rnn = nn.RNN(input_size, hidden_layer_size, batch_first=True)
        self.fc = nn.Linear(hidden_layer_size, output_size)

    def forward(self, x):
        x = self.embedding(x)
        out = self.rnn(x)[0]
        final_memory = torch.max(out, dim=1)[0]
        
        return self.fc(final_memory)


In [7]:
# Training the model

pad_id = tokenizer.pad_token_id

my_model = RNN(vocab_size=len(tokenizer), input_size=64, hidden_layer_size=128, output_size=3, pad_idx=pad_id).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(my_model.parameters(), lr=0.0005) 
num_epochs = 5

my_model.train()
for epoch in range(num_epochs):
    for batch in train_dataset:
        inputs, labels = batch
        outputs = my_model(inputs)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    print(f'Loss in Epoch {epoch+1} : {loss.item():.6f}')

Loss in Epoch 1 : 0.759785
Loss in Epoch 2 : 0.467606
Loss in Epoch 3 : 0.658773
Loss in Epoch 4 : 0.268289
Loss in Epoch 5 : 0.331197


In [8]:
# Testing the RNN
my_model.eval()
correct = 0
total = 0
for batch in test_dataset:
    inputs, labels = batch
    inputs, labels = inputs.to(device), labels.to(device)
    outputs = my_model(inputs)
    predicted = torch.max(outputs.data, 1)[1]
    total += labels.size(0)
    correct += (predicted == labels).sum().item()
print(f'Accuracy of the model: {100 * correct / total:.3f}%')

Accuracy of the model: 81.114%
